In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pywt
from sklearn.preprocessing import StandardScaler


In [78]:
df = pd.read_csv("../data/NF-UNSW-NB15-v3.csv")

## Prep data

In [79]:
df.shape

(2365424, 55)

In [80]:
df["Attack"].value_counts()

Attack
Benign            2237731
Exploits            42748
Fuzzers             33816
Generic             19651
Reconnaissance      17074
DoS                  5980
Backdoor             4659
Shellcode            2381
Analysis             1226
Worms                 158
Name: count, dtype: int64

## Analyse worms & benign to start

In [81]:
df_benign = df[df['Attack'] == 'Benign'].iloc[:5000].copy()
print("Benign rows:", len(df_benign))

Benign rows: 5000


## Remove non numeric columns

In [82]:
numerical_cols = df_benign.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [101]:
for col in  [
    'FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS',
    'IPV4_SRC_ADDR', 'IPV4_DST_ADDR',
    'L4_SRC_PORT', 'L4_DST_PORT', 'Label']: 
    if col in numerical_cols:
        numerical_cols.remove(col)

print("Number of numerical features:", len(numerical_cols))

Number of numerical features: 47


## Clean NaNs and infinities

In [102]:
df_benign[numerical_cols] = df_benign[numerical_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

## Standardise

In [103]:
scaler = StandardScaler()
X = StandardScaler().fit_transform(df_benign[numerical_cols])

## Apply CWT

In [104]:
scales = np.arange(1, 32)
wavelet = 'cmor3.5-1'

In [105]:
mean_mags = []
mean_phases = []

In [106]:
for i in range(X.shape[1]):
    signal = X[:, i]
    
    coeffs, _ = pywt.cwt(signal, scales, wavelet)
    
    magnitude = np.abs(coeffs)
    phase = np.angle(coeffs)
    
    mean_mags.append(magnitude.mean())
    mean_phases.append(np.abs(phase).mean())

## Inspect Results

In [108]:
cwt_summary = pd.DataFrame({
    'Feature': numerical_cols,
    'Mean_Magnitude': mean_mags,
    'Mean_Phase': mean_phases
})

cwt_summary.head(50)

,Feature,Mean_Magnitude,Mean_Phase
0,PROTOCOL,0.381641,1.602724
1,L7_PROTO,0.392191,1.589592
2,IN_BYTES,0.220833,1.570746
3,IN_PKTS,0.401259,1.571185
4,OUT_BYTES,0.374032,1.571554
5,OUT_PKTS,0.392864,1.570140
6,TCP_FLAGS,0.397485,1.594445
7,CLIENT_TCP_FLAGS,0.401285,1.594782
8,SERVER_TCP_FLAGS,0.403083,1.573436
9,FLOW_DURATION_MILLISECONDS,0.267708,1.571222


In [109]:
df_cwt_benign = df_benign[numerical_cols].copy()

for col in numerical_cols:
    df_cwt_benign[f'{col}_cwt_mag'] = cwt_summary.loc[cwt_summary['Feature'] == col, 'Mean_Magnitude'].values[0]
    df_cwt_benign[f'{col}_cwt_phase'] = cwt_summary.loc[cwt_summary['Feature'] == col, 'Mean_Phase'].values[0]

df_cwt_benign['Label'] = 0

df_cwt_benign.head()

,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,SERVER_TCP_FLAGS,FLOW_DURATION_MILLISECONDS,...,SRC_TO_DST_IAT_STDDEV_cwt_phase,DST_TO_SRC_IAT_MIN_cwt_mag,DST_TO_SRC_IAT_MIN_cwt_phase,DST_TO_SRC_IAT_MAX_cwt_mag,DST_TO_SRC_IAT_MAX_cwt_phase,DST_TO_SRC_IAT_AVG_cwt_mag,DST_TO_SRC_IAT_AVG_cwt_phase,DST_TO_SRC_IAT_STDDEV_cwt_mag,DST_TO_SRC_IAT_STDDEV_cwt_phase,Label
0,17,5.0,146,2,178,2,0,0,0,2,...,1.569942,0.0,3.141593,0.364918,1.569778,0.343435,1.570327,0.382536,1.569131,0
1,6,11.0,4704,28,2976,28,27,27,27,335,...,1.569942,0.0,3.141593,0.364918,1.569778,0.343435,1.570327,0.382536,1.569131,0
2,6,37.0,13662,238,548216,438,27,27,27,2460,...,1.569942,0.0,3.141593,0.364918,1.569778,0.343435,1.570327,0.382536,1.569131,0
3,17,5.0,146,2,178,2,0,0,0,1,...,1.569942,0.0,3.141593,0.364918,1.569778,0.343435,1.570327,0.382536,1.569131,0
4,17,5.0,130,2,162,2,0,0,0,1,...,1.569942,0.0,3.141593,0.364918,1.569778,0.343435,1.570327,0.382536,1.569131,0


## Complete for all attack types

In [ ]:
max_rows = 5000

In [ ]:
all_cwt_dfs = []

In [ ]:
attack_types = df['Attack'].unique()
attack_types = [x for x in attack_types if x != 'Benign'| 'Worm']